# Titanic Dataset

Most of the code cells below include comments explaining the task to be performed in those cells. Please delete the comments and add code to perform those tasks. There are a few code cells in which code has already been provided for you. In some cases, you will need to complete this code. 

# Import Statements

In [ ]:
# Import numpy, pandas, and matplotlib using the standard aliases. 
# Import the following tools from sklearn: 
#     Pipeline, SimpleImputer, ColumnTransformer, OneHotEncoder, StandardScaler
#     LogisticRegression, DecisionTreeClassifier, RandomForestClassifier, GridSearchCV

# Data manipulation and analysis
import pandas as pd
import numpy as np

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Machine Learning
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.ensemble import RandomForestClassifier, VotingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.metrics import classification_report, confusion_matrix, f1_score, accuracy_score
import xgboost as xgb

# Warnings
import warnings
warnings.filterwarnings('ignore')

# Set random seed for reproducibility
np.random.seed(42)


# Import joblib
import joblib

# Load Training Data

In [ ]:
# Load the training data into a DataFrame named 'train'. 
# Print the shape of the resulting DataFrame. 
# You do not need the test data in this notebook. 
# Load the dataset
train = pd.read_csv('/kaggle/input/titanic/train.csv')

train = train.sample(frac=1, random_state=1)

print("Dataset shape:", train.shape)
print("\nFirst few rows:")

print(train.shape)

In [ ]:
mb = train.memory_usage(index=True, deep=True).sum() / 1024**2
print(mb)

In [ ]:
# Display the head of the train DataFrame. 
train.head()

# Check for Missing Values

In [ ]:
train.isnull().sum().to_frame().T

In [ ]:
# Calculate and print the number of number of missing values in each column.
print(train.isnull().sum())

# Check Label Distribution

In [ ]:
# Display a DataFrame showing the proportion of observations with each 
# possible of the target variable (which is Survived).
(train.Survived.value_counts() / len(train)).to_frame()

# Preprocessing

In [ ]:
# We will start with some feature engineering. 

# Add a new column named 'FamSize' to the DataFrame. 
# This should be the sum of the 'SibSp' and 'Parch' columns. 
train['FamSize'] = train['SibSp'] + train['Parch']

# We will use the function below to determine the deck letter for each passenger:
def set_deck(cabin):
    if str(cabin) == 'nan':
        return 'Missing'
    return cabin[0]

# Use the map() method of the train DataFrame to apply the function above 
# to the 'Cabin' column. Store the results in a new column named 'Deck'. 
train['Deck'] = train['Cabin'].map(set_deck)

In [ ]:
# Create a list of numberical feature names. Use the following features: 'Age', 'FamSize', 'Fare'
num_features = ['Age', 'FamSize', 'Fare']

# Create a list of categorical feature names. Use the following features: 'Sex', 'Pclass', 'Deck', 'Embarked'
cat_features = ['Sex', 'Pclass', 'Deck', 'Embarked']

# Combine the two previous lists into one list named 'features'
features = num_features + cat_features

# Create a Pipeline object for processing the numerical features. 
# This pipeline should consist of a SimpleImputer and a StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer

num_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='mean')),
    ('scaler', StandardScaler())
])

# Create a Pipeline object for processing the categorical features. 
# This pipeline should consist of a SimpleImputer and a OneHotEncoder
cat_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='constant', fill_value='Missing')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

# Create a ColumnTransformer object that combines the two pipelines created above. 
# Name this ColumnTransformer 'preprocessor'
preprocessor = ColumnTransformer([
    ('num', num_transformer, num_features),
    ('cat', cat_transformer, cat_features)
])

In [ ]:
# Fit the preprocessor to the training data, selecting only the columns in the 'features' list. 
# Apply the fitted preprocessor to the training data, again selecting only the relevant columns. 
# Store the array created in the previous step into a variable named 'X_train'.
preprocessor.fit(train[features])
X_train = preprocessor.transform(train[features])

# Create a variable named 'y_train' that contains the training labels. 
y_train = train['Survived'].values

# Print the shapes of X_train and y_train.
print('X_train shape:', X_train.shape)
print('y_train shape:', y_train.shape)

# Model Selection

## Logistic Regression

In [ ]:
# Select a range of parameter values of C. 
# You might need to experiment with this to find a good value for C
# Update the code below to perform 10-Fold cross-validation. 
# Set the scoring parameter below to 'accuracy'

# %%time 

lr_clf = LogisticRegression(max_iter=1000, solver='saga', penalty='elasticnet')

lr_parameters = {
    'l1_ratio':[0, 0.5, 1],
    'C': [0.01, 0.1, 1, 10]
}

lr_grid = GridSearchCV(lr_clf, lr_parameters, cv=10, refit='True', n_jobs=-1, verbose=10, scoring='accuracy')
lr_grid.fit(X_train, y_train)

lr_model = lr_grid.best_estimator_

print('Best Parameters:', lr_grid.best_params_)
print('Best CV Score:  ', lr_grid.best_score_)
print('Training Acc:   ', lr_model.score(X_train, y_train))

In [ ]:
# Run this cell without any changes to view the CV results.

lr_summary = pd.DataFrame(lr_grid.cv_results_['params'])
lr_summary['cv_score'] = lr_grid.cv_results_['mean_test_score']

for r in lr_parameters['l1_ratio']:
    temp = lr_summary.query(f'l1_ratio == {r}')
    plt.plot(temp.C, temp.cv_score, label=r)
plt.xscale('log')
plt.ylim([0.75, 0.82])
plt.xlabel('Regularization Parameter (C)')
plt.ylabel('CV Score')
plt.legend(title='L1 Ratio', loc='lower right')
plt.grid()
plt.show()

print(lr_summary.to_string(index=False))

## Decicion Trees

In [ ]:
# Select values to consider for the max_depth and min_samples_leaf hyperparameters.
# You might need to experiment to find a good range of parameter values. 
# Update the code below to perform 10-Fold cross-validation. 
# Set the scoring parameter below to 'accuracy'

#%%time 

from sklearn.tree import DecisionTreeClassifier


dt_clf = DecisionTreeClassifier(random_state=1)

dt_parameters = {
    'max_depth': [3, 5, 7, 10, 15],
    'min_samples_leaf': [1, 2, 5, 10]
}

dt_grid = GridSearchCV(dt_clf, dt_parameters, cv=10, refit='True', n_jobs=-1, verbose=0, scoring='accuracy')
dt_grid.fit(X_train, y_train)

dt_model = dt_grid.best_estimator_

print('Best Parameters:', dt_grid.best_params_)
print('Best CV Score:  ', dt_grid.best_score_)
print('Training Acc:   ', dt_model.score(X_train, y_train))

In [ ]:
# Run this cell without any changes to view the CV results.

dt_summary = pd.DataFrame(dt_grid.cv_results_['params'])
dt_summary['cv_score'] = dt_grid.cv_results_['mean_test_score']

for ms in dt_parameters['min_samples_leaf']:
    temp = dt_summary.query(f'min_samples_leaf == {ms}')
    plt.plot(temp.max_depth, temp.cv_score, label=ms)
plt.xlabel('Maximum Depth')
plt.ylabel('CV Score')
plt.legend(title='Min Samples')
plt.grid()
plt.show()

print(dt_summary.to_string(index=False))

## Random Forests

In [ ]:
# Select a number of trees to use in your random forest.
# Select values to consider for the max_depth and min_samples_leaf hyperparameters.
# You might need to experiment to find a good range of parameter values. 
# Update the code below to perform 10-Fold cross-validation. 
# Set the scoring parameter below to 'accuracy'

# %%time 

rf_clf = RandomForestClassifier(random_state=1, n_estimators=100)

rf_parameters = {
    'max_depth': [3, 5, 7, 10, 15],
    'min_samples_leaf': [1, 2, 5, 10]
}

rf_grid = GridSearchCV(rf_clf, rf_parameters, cv=10, refit='True', n_jobs=-1, verbose=0, scoring='accuracy')
rf_grid.fit(X_train, y_train)

rf_model = rf_grid.best_estimator_

print('Best Parameters:', rf_grid.best_params_)
print('Best CV Score:  ', rf_grid.best_score_)
print('Training Acc:   ', rf_model.score(X_train, y_train))

In [ ]:
# Run this cell without any changes to view the CV results.

rf_summary = pd.DataFrame(rf_grid.cv_results_['params'])
rf_summary['cv_score'] = rf_grid.cv_results_['mean_test_score']

for ms in rf_parameters['min_samples_leaf']:
    temp = rf_summary.query(f'min_samples_leaf == {ms}')
    plt.plot(temp.max_depth, temp.cv_score, label=ms)
plt.xlabel('Maximum Depth')
plt.ylabel('CV Score')
plt.legend(title='Min Samples')
plt.grid()
plt.show()

print(rf_summary.to_string(index=False))

# Save Pipeline and Model

In [ ]:
# Save your pipeline to a file. 
# Determine the best model found above and save that to a file. 
# Download both files to your local device and then upload them as a Kaggle dataset. 
joblib.dump(preprocessor, 'titanic_preprocessor.joblib')

# Determine best model (compare CV scores)
print('Logistic Regression CV Score:', lr_grid.best_score_)
print('Decision Tree CV Score:', dt_grid.best_score_)
print('Random Forest CV Score:', rf_grid.best_score_)

# Save the best performing model
best_models = [('lr', lr_grid.best_score_, lr_model), 
               ('dt', dt_grid.best_score_, dt_model), 
               ('rf', rf_grid.best_score_, rf_model)]
best_model_name, best_score, best_model = max(best_models, key=lambda x: x[1])

print(f'Best model: {best_model_name} with CV score: {best_score}')
joblib.dump(best_model, 'titanic_best_model.joblib')
print('Files saved successfully!')